# Kestrel Home: Warranty Claim Fraud

## 1. Import libraries

## 0. Install libraries

Run once per environment. Versions in the comment are the ones this notebook was run with.

In [ ]:
# Tested with: numpy 2.1.3, pandas 2.2.3, matplotlib 3.10.0, seaborn 0.13.2,
# scikit-learn 1.6.1, xgboost 3.1.3, scipy 1.15.3, statsmodels 0.14.4
%pip install -q numpy pandas matplotlib seaborn scikit-learn xgboost scipy statsmodels

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

## 2. Load all CSV files

File names start with a UUID prefix, so each file is found by its suffix. The `Data` folder is found by walking up from the notebook's folder.

In [ ]:
DATA_DIR = next(p / "Data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data").is_dir())

def find_file(suffix):
    matches = list(DATA_DIR.glob(f"*{suffix}"))
    assert len(matches) == 1, f"Expected 1 file ending in {suffix}, found {len(matches)}"
    return matches[0]

train = pd.read_csv(find_file("train.csv"), parse_dates=["submitted_at"])
test = pd.read_csv(find_file("test_unlabelled.csv"), parse_dates=["submitted_at"])
partners = pd.read_csv(find_file("partners.csv"), parse_dates=["onboarded_date"])
products = pd.read_csv(find_file("products.csv"))
sample_submission = pd.read_csv(find_file("sample_submission.csv"))

print("Data folder:", DATA_DIR)
for name, df in {"train": train, "test": test, "partners": partners,
                 "products": products, "sample_submission": sample_submission}.items():
    print(f"{name:<18} {df.shape}")

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
partners.head()

In [ ]:
products.head()

In [ ]:
sample_submission.head()

## 3. Understand the target: `is_fraud`

`is_fraud` is 1 = fraud, 0 = not fraud, blank = investigation still open at export (train only). Tanmay's email gives three warnings that all affect the target:
1. The same claim can appear twice (re-submitted after being bounced).
2. Open cases are blank in the CRM, but Zoho couldn't store blanks, so old open cases came across as **0**.
3. Serials are messy (this matters for features, not for the target).

### 3.1 Raw label counts

In [ ]:
counts = train["is_fraud"].value_counts(dropna=False).rename(index={1.0: "fraud (1)", 0.0: "not fraud (0)"}).fillna(0)
counts.index = counts.index.fillna("blank (undecided)")
pd.DataFrame({"rows": counts, "share_%": (counts / len(train) * 100).round(2)})

### 3.2 Duplicate claim IDs

Check how many claims repeat, and whether the copies of one claim ever disagree on the label.

In [ ]:
dup_mask = train["claim_id"].duplicated(keep=False)
print("Rows that share a claim_id :", dup_mask.sum())
print("Extra (repeated) rows      :", train["claim_id"].duplicated().sum())
print("Unique claims              :", train["claim_id"].nunique())

labels_per_claim = train[dup_mask].groupby("claim_id")["is_fraud"].nunique(dropna=False)
print("Duplicated claims with conflicting labels:", (labels_per_claim > 1).sum())

# Keep the first submission of each claim (same rule as the cleaning in Section 4)
train_dedup = train.sort_values("submitted_at").drop_duplicates("claim_id", keep="first").reset_index(drop=True)
print("Rows after de-duplication  :", len(train_dedup))

### 3.3 Blank labels by source system

If blanks only appear in `crm`, then the open cases in `legacy_zoho` must be hiding as 0.

In [ ]:
by_source = train_dedup.groupby("source").agg(
    claims=("claim_id", "size"),
    fraud=("is_fraud", lambda s: (s == 1).sum()),
    not_fraud=("is_fraud", lambda s: (s == 0).sum()),
    blank=("is_fraud", lambda s: s.isna().sum()),
)
by_source["fraud_rate_%"] = (by_source["fraud"] / (by_source["fraud"] + by_source["not_fraud"]) * 100).round(2)
by_source

### 3.4 Label status over time

Blank labels can only exist from the CRM go-live (Oct 2025). If they are a steady share every month, the Zoho months probably hid a similar share of open cases as 0.

In [ ]:
status = train_dedup["is_fraud"].map({1.0: "fraud", 0.0: "not fraud"}).fillna("blank")
monthly = pd.crosstab(train_dedup["submitted_at"].dt.to_period("M"), status)
monthly["blank_%"] = (monthly["blank"] / monthly.sum(axis=1) * 100).round(1)
monthly["fraud_rate_%"] = (monthly["fraud"] / (monthly["fraud"] + monthly["not fraud"]) * 100).round(2)
monthly

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
monthly["fraud_rate_%"].plot(ax=ax, marker="o", label="fraud rate % (labelled claims)")
monthly["blank_%"].plot(ax=ax, marker="s", label="blank label %")
ax.axvline(pd.Period("2025-10", "M"), color="grey", ls="--")
ax.text(pd.Period("2025-10", "M"), ax.get_ylim()[1] * 0.92, " CRM go-live (1 Oct 2025)", color="grey")
ax.axvline(pd.Period("2026-05", "M"), color="red", ls="--")
ax.text(pd.Period("2026-05", "M"), ax.get_ylim()[1] * 0.80, " auto-approve < Rs 2,000", color="red")
ax.set_ylabel("%"); ax.set_xlabel(""); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

### 3.5 Labelled set used for modelling

Drop the blank (undecided) claims. They are unknown, not "not fraud".

In [ ]:
labelled = train_dedup[train_dedup["is_fraud"].notna()].copy()
labelled["is_fraud"] = labelled["is_fraud"].astype(int)

n_fraud = labelled["is_fraud"].sum()
print(f"Labelled claims : {len(labelled):,}")
print(f"Fraud claims    : {n_fraud:,}  ({n_fraud / len(labelled):.2%})")
print(f"Not fraud       : {len(labelled) - n_fraud:,}")
print(f"Imbalance       : about 1 fraud per {round((len(labelled) - n_fraud) / n_fraud)} genuine claims")

### 3.6 Why "accuracy above 97%" is the wrong KPI

A model that says **"not fraud" for every claim** catches nothing, yet its accuracy is:

In [ ]:
all_zero_accuracy = (labelled["is_fraud"] == 0).mean()
fraud_rupees = labelled.loc[labelled["is_fraud"] == 1, "claim_amount_inr"].sum()
print(f"Accuracy of predicting 'not fraud' for everything: {all_zero_accuracy:.2%}")
print(f"Fraud it catches: 0 of {n_fraud} claims, Rs 0 of Rs {fraud_rupees:,.0f}")

### 3.7 The target in rupees

Finance cares about **₹ of fraud stopped per claim reviewed**, so the claim amount matters as much as the label.

In [ ]:
amt = labelled.groupby("is_fraud")["claim_amount_inr"].describe()[["count", "mean", "50%", "max"]]
amt["total_Rs"] = labelled.groupby("is_fraud")["claim_amount_inr"].sum()
amt.rename(index={0: "not fraud", 1: "fraud"}).round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(data=labelled, x="claim_amount_inr", hue="is_fraud", stat="density", common_norm=False,
             log_scale=True, bins=40, element="step", ax=ax)
ax.axvline(2000, color="red", ls="--"); ax.text(2000, ax.get_ylim()[1] * 0.9, " Rs 2,000", color="red")
ax.set_title("Claim amount: fraud vs not fraud (log scale)")
plt.tight_layout(); plt.show()

### 3.8 The May 2026 rule change shifts the target

From 1 May 2026, claims under ₹2,000 are auto-approved without inspection. The whole test set (Jul–Sep 2026) comes after this change.

In [ ]:
labelled["period"] = np.where(labelled["submitted_at"] >= "2026-05-01", "after 1 May 2026", "before 1 May 2026")
labelled["size"] = np.where(labelled["claim_amount_inr"] < 2000, "small (< Rs 2,000)", "large (>= Rs 2,000)")
shift = labelled.pivot_table(index="period", columns="size", values="is_fraud", aggfunc=["mean", "sum", "size"])
shift["mean"] = (shift["mean"] * 100).round(2)
shift.rename(columns={"mean": "fraud_rate_%", "sum": "frauds", "size": "claims"}).iloc[::-1]

### 3.9 What we learned about the target

| Finding | Number | What it means for modelling |
|---|---|---|
| Duplicate claim IDs | 681 extra rows, 0 label conflicts | De-duplicate (keep the first submission) → **11,348** unique claims. Split train/validation by claim, never by row. |
| Blank labels | 202 claims, **all in `crm`**, about **3% every month** since Oct 2025 | Drop them. They are unknown, not "not fraud". |
| Hidden blanks in Zoho | 0 blanks in `legacy_zoho` | If ~3% of Zoho cases were also open, roughly **140 Zoho "0"s are really unknown**. Treat Zoho negatives as noisy (e.g. lower sample weight or a sensitivity check). |
| Class balance | **141 fraud of 11,146 labelled (1.27%)**, about 1 in 78 | Very imbalanced. Use ranking metrics, not a 0.5 threshold. |
| Accuracy KPI | "Predict nothing is fraud" = **98.73% accuracy**, catches ₹0 of ₹6.38 lakh | The board's ">97% accuracy" is met by a useless model. Report Finance's **₹ fraud stopped per claim reviewed** instead. |
| Fraud claim size | Fraud median ₹2,265 vs genuine ₹1,431 | Claim amount is a signal, and it sets the value of catching each fraud. |
| May 2026 shift | Small-claim fraud rate **0.54% → 3.23%**, large-claim **1.93% → 0.30%**. Monthly fraud rate 1.97% (May), **3.09%** (Jun) | Fraud moved to auto-approved small claims. The test set is entirely after the change, so recent months matter most. Validate on May–Jun 2026. |

## 4. Data cleaning

Issues found while checking every column (train + test):

| Column | Issue | Fix |
|---|---|---|
| `claim_id` | 681 repeated rows in train. Copies are identical except `submitted_at` (the re-submission comes days later). Test has no repeats. | Keep the **first** submission: that is when the claim was really made. |
| `is_fraud` | 202 blank (open) claims, all in `crm` | Drop from training. Flag Zoho `0`s as possibly noisy. |
| `product_serial` | 4 formats: `KH123456789`, `kh123456789`, `␣KH123456789`, `KH-123456789` | Upper-case and remove non-alphanumerics → 100% become `KH` + 9 digits. Keep the raw format as a column, since how a partner types serials may be a signal. |
| `photo_attached`, `partner_inspected` | `Y`/`N` text | Convert to 1/0. |
| `claim_description` | 12 standard fault phrases, but 4 claims have extra text appended after the fault, addressed to automated analysis tools rather than describing the fault | Keep only the fault phrase (text before the first `.` or `;`), lower-case. One row says `display blank`, mapped to `display not working`. |
| `inspector_note` | Blank when not inspected, and also blank on ~12% of inspected claims | Fill with `"no note"` and add a `has_inspector_note` flag. |
| `partner_inspected` | From 1 May 2026, claims < ₹2,000 are mostly not inspected (policy change). A few large claims after May are also not inspected. | Keep as is, add `after_may_change` and `small_claim` flags so the model can tell the regimes apart. |
| Joins | All `partner_id` and `sku` values exist in the lookup tables. No claim before its partner's onboarding date. No claim beyond warranty or above list price. | Merge partners and products. Nothing to fix. |

In [ ]:
SERIAL_OK = r"KH\d{9}"
POLICY_CHANGE = pd.Timestamp("2026-05-01")

def serial_format(s):
    if s.startswith(" "):
        return "leading_space"
    if "-" in s:
        return "hyphen"
    if s[:2].islower():
        return "lowercase"
    return "clean"

def clean_claims(df):
    out = df.copy()

    # Serial: normalise, but remember how it was typed
    out["serial_format"] = out["product_serial"].astype(str).map(serial_format)
    out["product_serial"] = out["product_serial"].astype(str).str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)

    # Y/N -> 1/0
    for col in ["photo_attached", "partner_inspected"]:
        out[col] = out[col].map({"Y": 1, "N": 0}).astype(int)

    # Description: keep only the standard fault phrase
    out["claim_description"] = (out["claim_description"].str.lower()
                                .str.split(r"[.;]", regex=True).str[0].str.strip()
                                .replace({"display blank": "display not working"}))

    # Inspector note
    out["has_inspector_note"] = out["inspector_note"].notna().astype(int)
    out["inspector_note"] = out["inspector_note"].fillna("no note")

    # Policy regime flags
    out["after_may_change"] = (out["submitted_at"] >= POLICY_CHANGE).astype(int)
    out["small_claim"] = (out["claim_amount_inr"] < 2000).astype(int)

    # Lookups
    out = (out.merge(partners, on="partner_id", how="left", validate="many_to_one")
              .merge(products, on="sku", how="left", validate="many_to_one"))
    return out

### 4.1 Clean train and test

In [ ]:
# All unique claims (first submission), INCLUDING open investigations.
# Used to build partner-behaviour features, so train counts claims the same way as test.
train_all_clean = clean_claims(train.sort_values("submitted_at").drop_duplicates("claim_id", keep="first"))
train_all_clean = train_all_clean.sort_values("submitted_at").reset_index(drop=True)

# Zoho could not store blanks, so some of its 0s are really open cases
train_all_clean["label_maybe_noisy"] = ((train_all_clean["source"] == "legacy_zoho")
                                        & (train_all_clean["is_fraud"] == 0)).astype(int)

# Labelled claims only: used for training and evaluation
train_clean = train_all_clean[train_all_clean["is_fraud"].notna()].reset_index(drop=True)
train_clean["is_fraud"] = train_clean["is_fraud"].astype(int)
n_blank = len(train_all_clean) - len(train_clean)

test_clean = clean_claims(test).sort_values("submitted_at").reset_index(drop=True)

print(f"train: {len(train):,} raw rows -> {len(train_all_clean):,} unique claims (train_all_clean) "
      f"-> {len(train_clean):,} labelled (train_clean, {n_blank} open claims set aside)")
print(f"test : {len(test):,} raw rows -> {len(test_clean):,}")

### 4.2 Check the result

In [ ]:
for name, df in [("train_clean", train_clean), ("test_clean", test_clean)]:
    assert df["claim_id"].is_unique, f"{name}: duplicate claim_id"
    assert df["product_serial"].str.fullmatch(SERIAL_OK).all(), f"{name}: bad serial"
    assert df[["city", "family", "list_price_inr"]].notna().all().all(), f"{name}: failed join"
    assert df.drop(columns=["is_fraud"], errors="ignore").isna().sum().sum() == 0, f"{name}: missing values"
assert set(test_clean["claim_id"]) == set(sample_submission["claim_id"]), "test ids differ from sample_submission"
assert not set(train_clean["claim_id"]) & set(test_clean["claim_id"]), "claim in both train and test"
print("All checks passed")

In [ ]:
pd.DataFrame({
    "serial_format (train)": train_clean["serial_format"].value_counts(),
    "serial_format (test)": test_clean["serial_format"].value_counts(),
})

In [ ]:
pd.DataFrame({
    "train": train_clean["claim_description"].value_counts(),
    "test": test_clean["claim_description"].value_counts(),
})

In [ ]:
print("Train columns:", list(train_clean.columns))
train_clean.head()

### 4.3 Cleaning summary

- **Train:** 12,029 raw rows → 11,348 unique claims → **11,146** labelled claims (202 open cases dropped).
- **Test:** 2,252 claims, no duplicates; IDs match `sample_submission` exactly.
- Serials all normalised to `KH` + 9 digits; the original typing style is kept in `serial_format`.
- No missing values left, every claim joined to its partner and product.
- `train_all_clean` keeps all 11,348 unique claims, including the 202 open ones. Partner-behaviour features are built from it, so claim counts in train match how test claims are counted. `train_clean` (labelled only) is used for training and evaluation.
- `label_maybe_noisy` marks the Zoho `0`s, to be used later for down-weighting or a sensitivity check.

Next step: feature engineering (partner history, claim amount vs product price, timing).

## 5. How the tables relate

```
partners.csv (380 rows)            train / test claims             products.csv (21 rows)
-----------------------            -------------------             ----------------------
partner_id  (PK)  <---- N : 1 ---- partner_id                      sku  (PK)
city                               sku          ---- N : 1 ---->   family
onboarded_date                     claim_amount_inr                list_price_inr
partner_type                       submitted_at ...                warranty_months
```

- Each claim belongs to **one partner** and **one product (SKU)**. A partner or a SKU has many claims.
- `partners` and `products` are lookup tables. They carry no label, so all fraud signal from them comes through the claims.
- Useful combinations: `claim_amount_inr / list_price_inr` (how much of the product's value is claimed), `days_since_purchase / warranty_months` (how far into the warranty), and `submitted_at - onboarded_date` (how long the partner had been with Kestrel at claim time).

### 5.1 Coverage and cardinality

In [ ]:
train_p, test_p = set(train_clean["partner_id"]), set(test_clean["partner_id"])
print(f"Partners: {len(partners)} total | {len(train_p)} in train | {len(test_p)} in test | "
      f"{len(test_p - train_p)} appear only in test (no history)")
print(f"SKUs    : {len(products)} total | {train_clean['sku'].nunique()} in train | {test_clean['sku'].nunique()} in test")
print("Claims per partner (train):", train_clean.groupby("partner_id").size().describe()[["min", "50%", "max"]].to_dict())
print("Partners onboarded during the test window:", (partners["onboarded_date"] >= "2026-07-01").sum())

### 5.2 Fraud rate by partner and product attributes

In [ ]:
def fraud_rate(df, col):
    return (df.groupby(col, observed=True)["is_fraud"]
              .agg(fraud_rate_pct=lambda s: round(s.mean() * 100, 2), frauds="sum", claims="size")
              .sort_values("fraud_rate_pct", ascending=False))

display(fraud_rate(train_clean, "partner_type"))
display(fraud_rate(train_clean, "family"))
display(fraud_rate(train_clean, "city").head(8))

In [ ]:
tenure_days = (train_clean["submitted_at"] - train_clean["onboarded_date"]).dt.days
tenure_band = pd.cut(tenure_days, [0, 90, 180, 365, 730, 10_000], right=False,
                     labels=["< 3 m", "3-6 m", "6-12 m", "1-2 y", "> 2 y"])
period = np.where(train_clean["after_may_change"] == 1, "after 1 May 2026", "before 1 May 2026")
tab = (train_clean.assign(tenure=tenure_band, period=period)
       .pivot_table(index="tenure", columns="period", values="is_fraud", aggfunc=["mean", "sum", "size"], observed=True))
tab["mean"] = (tab["mean"] * 100).round(2)
tab.rename(columns={"mean": "fraud_rate_%", "sum": "frauds", "size": "claims"})

### 5.3 Is fraud spread across partners or concentrated in a few?

In [ ]:
fraud_by_partner = train_clean[train_clean["is_fraud"] == 1].groupby("partner_id").size().sort_values(ascending=False)
print(f"Partners with at least one fraud: {len(fraud_by_partner)} of {len(train_p)}")
print(f"Top 10 partners hold {fraud_by_partner.head(10).sum() / fraud_by_partner.sum():.0%} of all fraud")

pre = set(train_clean.query("is_fraud == 1 and after_may_change == 0")["partner_id"])
post = set(train_clean.query("is_fraud == 1 and after_may_change == 1")["partner_id"])
print(f"Fraud partners before May: {len(pre)} | after May: {len(post)} | in both: {len(pre & post)}")

new_partners = partners.loc[partners["onboarded_date"] >= "2025-07-01", "partner_id"]
new_with_claims = set(new_partners) & train_p
print(f"New partners (onboarded since Jul 2025) with claims: {len(new_with_claims)} | "
      f"with any fraud: {len(new_with_claims & set(fraud_by_partner.index))}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
top = fraud_by_partner.head(20)
colors = ["tab:red" if p in post else "tab:blue" for p in top.index]
ax.bar(top.index, top.values, color=colors)
ax.set_title("Top 20 partners by fraud count (red = fraud after 1 May 2026, blue = before)")
ax.set_ylabel("fraud claims"); ax.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

### 5.4 What the relationships tell us

- **Partner type:** franchises have about 2x the fraud rate of service centres; freelancers have the lowest.
- **Product:** robot vacuums and water purifiers (the most expensive) are slightly higher, but family is a weak signal on its own.
- **Fraud is concentrated:** only 51 of 366 partners have any fraud, and the top 10 hold 63% of it.
- **The fraudsters changed in May 2026:** the partners committing fraud after the rule change are a completely different set from those before. Fraud history from 2025 will not, by itself, find the new ones.
- **New partners, the nuance:** after May almost all fraud comes from partners with < 1 year tenure. But most new partners have no fraud at all, and their claims are mostly genuine. Tenure helps rank claims, but flagging every new partner would swamp the 40-claims-a-month desk with genuine customers. This supports both Ritu (new partners matter) and Meenal (do not paint them all with one brush).

## 6. Feature engineering

**Rule: every feature must be known at the moment the claim is submitted.** Anything about a partner's history uses only claims submitted *before* the current one. Train and test claims are combined so that test claims also get history from the train period.

**Label lag:** a fraud outcome is only known after investigation. Label-based history therefore uses only claims submitted **at least 30 days before** the current claim (`LABEL_LAG_DAYS`). Without this lag, training claims would see outcomes from the day before, while test claims only see outcomes up to 30 June. The model would learn to trust partner history more than it deserves (train-serve skew).

| Group | Features |
|---|---|
| Claim | `log_amount`, `amount_to_price`, `near_threshold` (Rs 1,800-1,999, just under the auto-approve limit), `warranty_used_frac`, `days_since_purchase`, `customer_prior_claims`, `photo_attached`, `partner_inspected`, `has_inspector_note`, `small_claim`, `after_may_change`, `submit_hour`, `submit_dow` |
| Partner profile | `partner_type`, `partner_tenure_days`, `partner_is_new` (< 1 year) |
| Partner behaviour (last 30 / 90 days, no labels) | claim count, share of near-threshold claims, share not inspected, mean `amount_to_price` |
| Partner fraud history (labels, 30-day lag) | `p_past_frauds`, `p_past_labelled`, smoothed `p_past_fraud_rate`, `p_frauds_90d` (frauds among claims submitted 30–90 days earlier, because the fraudsters changed) |
| Serial | `serial_prior_claims` (earlier claims on the same serial) |
| Categorical | `family`, `claim_description`, `inspector_note`, `serial_format`, `partner_type` |

In [ ]:
LABEL_LAG_DAYS = 30   # an investigation outcome is assumed known 30 days after the claim


def partner_window_features(claims, days):
    """Partner behaviour in the `days` before each claim (current claim excluded). Uses no labels."""
    window = f"{days}D"
    d = claims.set_index("submitted_at")
    g = d.groupby("partner_id")
    roll = lambda col, how: g[col].transform(lambda s: getattr(s.rolling(window, closed="left"), how)())
    return pd.DataFrame({
        "claim_id": d["claim_id"].values,
        f"p_claims_{days}d": roll("claim_id", "count").values,
        f"p_near_threshold_share_{days}d": roll("near_threshold", "mean").values,
        f"p_uninspected_share_{days}d": roll("not_inspected", "mean").values,
        f"p_amount_ratio_mean_{days}d": roll("amount_to_price", "mean").values,
    })


def labels_before(claims, when):
    """For each claim: the partner's labelled claims and frauds among claims submitted strictly before `when`."""
    events = claims.loc[claims["is_fraud"].notna(), ["partner_id", "submitted_at", "is_fraud"]].sort_values("submitted_at")
    events["cum_frauds"] = events.groupby("partner_id")["is_fraud"].cumsum()
    events["cum_labelled"] = events.groupby("partner_id").cumcount() + 1
    query = pd.DataFrame({"row": claims.index, "partner_id": claims["partner_id"], "when": when}).sort_values("when")
    found = pd.merge_asof(query, events.drop(columns="is_fraud"), left_on="when", right_on="submitted_at",
                          by="partner_id", allow_exact_matches=False)
    found = found.set_index("row").reindex(claims.index)
    return found["cum_frauds"].fillna(0).values, found["cum_labelled"].fillna(0).values


def build_features(train_claims, test_claims, label_lag_days=LABEL_LAG_DAYS, prior_rate=0.0127, prior_weight=20):
    """`train_claims` should include open (unlabelled) claims, so partner claim counts match the test set."""
    claims = pd.concat([train_claims.assign(dataset="train"), test_claims.assign(dataset="test")], ignore_index=True)
    claims = claims.sort_values(["submitted_at", "claim_id"]).reset_index(drop=True)

    # Claim level
    claims["log_amount"] = np.log1p(claims["claim_amount_inr"])
    claims["amount_to_price"] = claims["claim_amount_inr"] / claims["list_price_inr"]
    claims["near_threshold"] = claims["claim_amount_inr"].between(1800, 1999.99).astype(int)
    claims["warranty_used_frac"] = claims["days_since_purchase"] / (claims["warranty_months"] * 30.44)
    claims["not_inspected"] = 1 - claims["partner_inspected"]
    claims["submit_hour"] = claims["submitted_at"].dt.hour
    claims["submit_dow"] = claims["submitted_at"].dt.dayofweek

    # Partner profile at claim time
    claims["partner_tenure_days"] = (claims["submitted_at"] - claims["onboarded_date"]).dt.days
    claims["partner_is_new"] = (claims["partner_tenure_days"] < 365).astype(int)

    # Partner behaviour in trailing windows (all claims count, labelled or not)
    for days in (30, 90):
        claims = claims.merge(partner_window_features(claims, days), on="claim_id", how="left")
    window_cols = [c for c in claims.columns if c.startswith("p_") and c.endswith(("_30d", "_90d"))]
    claims[window_cols] = claims[window_cols].fillna(0)

    # Partner fraud history: only outcomes of claims submitted at least `label_lag_days` earlier
    lag = pd.Timedelta(days=label_lag_days)
    frauds_lagged, labelled_lagged = labels_before(claims, claims["submitted_at"] - lag)
    frauds_90d_ago, _ = labels_before(claims, claims["submitted_at"] - pd.Timedelta(days=90))
    claims["p_past_frauds"] = frauds_lagged
    claims["p_past_labelled"] = labelled_lagged
    claims["p_frauds_90d"] = frauds_lagged - frauds_90d_ago          # frauds among claims 30-90 days old
    claims["p_past_fraud_rate"] = ((claims["p_past_frauds"] + prior_rate * prior_weight)
                                   / (claims["p_past_labelled"] + prior_weight))

    # Serial history
    claims["serial_prior_claims"] = claims.groupby("product_serial").cumcount()
    return claims

### 6.1 Build features

In [ ]:
claims_fe = build_features(train_all_clean, test_clean)

NUMERIC = [
    "log_amount", "amount_to_price", "near_threshold", "warranty_used_frac", "days_since_purchase",
    "customer_prior_claims", "photo_attached", "partner_inspected", "has_inspector_note",
    "small_claim", "after_may_change", "submit_hour", "submit_dow",
    "partner_tenure_days", "partner_is_new",
    "p_claims_30d", "p_near_threshold_share_30d", "p_uninspected_share_30d", "p_amount_ratio_mean_30d",
    "p_claims_90d", "p_near_threshold_share_90d", "p_uninspected_share_90d", "p_amount_ratio_mean_90d",
    "p_frauds_90d", "p_past_frauds", "p_past_labelled", "p_past_fraud_rate",
    "serial_prior_claims",
]
CATEGORICAL = ["partner_type", "family", "claim_description", "inspector_note", "serial_format"]
FEATURES = NUMERIC + CATEGORICAL

for col in CATEGORICAL:
    claims_fe[col] = claims_fe[col].astype("category")

# Train on labelled claims only (open claims were used for features, not as examples)
train_fe = claims_fe[(claims_fe["dataset"] == "train") & claims_fe["is_fraud"].notna()].reset_index(drop=True)
train_fe["is_fraud"] = train_fe["is_fraud"].astype(int)
test_fe = claims_fe[claims_fe["dataset"] == "test"].reset_index(drop=True)

assert len(train_fe) == len(train_clean) and len(test_fe) == len(test_clean)
assert claims_fe[FEATURES].isna().sum().sum() == 0, "missing values in features"
print(f"{len(FEATURES)} features ({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical)")
print(f"train_fe {train_fe.shape} | test_fe {test_fe.shape}")

### 6.2 Does each feature separate fraud? (train)

Fraud rate by feature band. A useful feature shows a clear rise or fall across bands.

In [ ]:
def band_rate(df, col, bins):
    bands = pd.cut(df[col], bins, right=False)
    return (df.groupby(bands, observed=True)["is_fraud"]
              .agg(fraud_rate_pct=lambda s: round(s.mean() * 100, 2), frauds="sum", claims="size"))

checks = {
    "amount_to_price": [0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.01],
    "customer_prior_claims": [0, 1, 2, 3, 10],
    "p_past_frauds": [0, 1, 2, 4, 100],
    "p_frauds_90d": [0, 1, 2, 100],
    "p_amount_ratio_mean_90d": [0, 0.25, 0.35, 0.45, 1.1],
    "p_claims_30d": [0, 3, 5, 8, 100],
    "partner_tenure_days": [0, 180, 365, 730, 10_000],
    "serial_prior_claims": [0, 1, 2, 10],
}
for col, bins in checks.items():
    print(f"--- {col}")
    display(band_rate(train_fe, col, bins))

In [ ]:
post_may = train_fe[train_fe["after_may_change"] == 1]
print("After 1 May 2026 only (the regime the test set is in):")
for col, bins in {"near_threshold": [0, 1, 2], "partner_is_new": [0, 1, 2],
                  "p_past_frauds": [0, 1, 2, 100], "p_near_threshold_share_90d": [0, 0.05, 0.15, 1.01]}.items():
    print(f"--- {col}")
    display(band_rate(post_may, col, bins))

### 6.3 Feature summary

**Strong signals**
- **Partner fraud history** (`p_past_frauds`, `p_frauds_90d`): claims from partners with confirmed frauds 30+ days earlier are many times more likely to be fraud.
- **`customer_prior_claims`**: the fraud rate rises steeply with each earlier claim by the customer.
- **`amount_to_price`**: claims for more than half the product's price are much riskier.
- **After May:** claims just under Rs 2,000 (`near_threshold`) and claims from partners < 1 year old carry most of the fraud.

**Weak or no signal:** product family, submission hour, `serial_prior_claims`. They are kept for now and the model will decide.

**Caveats**
- The partners committing fraud changed in May 2026, so all-time history over-weights old fraudsters. `p_frauds_90d` gives a recent view.
- Fraud labels arrive after investigation. Past-fraud features only use outcomes of claims at least 30 days old, the same rule for train and test. After May this weakens the history signal: the new outlets' frauds are recent, so many are not yet visible.
- 14 partners appear only in the test set and have no history. Their history features are 0.

### 6.4 Hypothesis-driven features

The hypothesis test (`Ritu_Hypothesis_New_Partners.ipynb`) showed that tenure matters **only after the May 2026 rule change**, and that small uninspected claims are where the new fraud happens. Two interaction features capture this directly:

| Feature | Definition | Why |
|---|---|---|
| `new_x_post` | `partner_is_new × after_may_change` | New-partner risk exists only in the post-May regime |
| `near_thr_x_uninsp` | `near_threshold × not_inspected` | Claims just under Rs 2,000 that skipped inspection |

In [ ]:
for df in (claims_fe, train_fe, test_fe):
    df["new_x_post"] = df["partner_is_new"] * df["after_may_change"]
    df["near_thr_x_uninsp"] = df["near_threshold"] * (1 - df["partner_inspected"])

for col in ["new_x_post", "near_thr_x_uninsp"]:
    print(f"--- {col}")
    display(band_rate(train_fe, col, [0, 1, 2]))

### 6.5 Feature strength

Univariate **AUC** of each numeric feature against `is_fraud`: 0.5 = no signal; further from 0.5 (in either direction) = stronger.
It is computed on all train claims **and** on post-May claims only, because the test set is entirely post-May and some signals changed after the rule.

In [ ]:
from sklearn.metrics import roc_auc_score

CANDIDATES = NUMERIC + ["new_x_post", "near_thr_x_uninsp"]
post_may = train_fe[train_fe["after_may_change"] == 1]

def auc(df, col):
    return roc_auc_score(df["is_fraud"], df[col]) if df[col].nunique() > 1 else np.nan

strength = pd.DataFrame({
    "AUC all": [auc(train_fe, c) for c in CANDIDATES],
    "AUC post-May": [auc(post_may, c) for c in CANDIDATES],
}, index=CANDIDATES)
strength["strength all"] = (strength["AUC all"] - 0.5).abs()
strength["strength post-May"] = (strength["AUC post-May"] - 0.5).abs()
strength = strength.sort_values("strength post-May", ascending=False).round(3)
strength

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
strength[["strength all", "strength post-May"]].iloc[::-1].plot.barh(ax=ax, width=0.8)
ax.axvline(0.05, color="grey", ls="--", lw=1)
ax.set(title="Feature strength |AUC - 0.5|", xlabel="strength (0 = no signal)")
plt.tight_layout(); plt.show()

**Result**
- **Partner fraud history** (`p_past_fraud_rate`, `p_frauds_90d`, `p_past_frauds`) is the strongest signal over all data (AUC ≈ 0.72–0.74). After May it is weaker, because with the 30-day lag the new outlets' recent frauds are not yet visible.
- **`new_x_post`**, **`partner_tenure_days`** and recent partner claim volume (`p_claims_90d`) are the strongest post-May signals, matching the hypothesis test.
- **Signal drift:** `amount_to_price`, `log_amount` and `p_amount_ratio_mean_90d` are strong over all data but close to useless after May, because fraud moved from large claims to small ones. `customer_prior_claims` also weakens after May.
- `serial_prior_claims`, `submit_hour` and `photo_attached` are weak in both periods.

### 6.6 Redundancy

Pairs of numeric features with |correlation| > 0.8 carry largely the same information. Keep the stronger one of each pair.

In [ ]:
corr = train_fe[CANDIDATES].corr().abs()
pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
             .rename("abs_corr").reset_index().query("abs_corr > 0.8")
             .sort_values("abs_corr", ascending=False))
pairs["stronger (post-May)"] = [a if strength.loc[a, "strength post-May"] >= strength.loc[b, "strength post-May"] else b
                                for a, b in zip(pairs["level_0"], pairs["level_1"])]
pairs.rename(columns={"level_0": "feature A", "level_1": "feature B"}).round(3).reset_index(drop=True)

### 6.7 Final feature set

| Dropped | Reason |
|---|---|
| `p_past_frauds` | 0.97 correlated with `p_past_fraud_rate`, which is the stronger of the two |
| `days_since_purchase` | 0.81 correlated with `warranty_used_frac`; equally strong, but `warranty_used_frac` also accounts for warranty length |
| `serial_prior_claims`, `submit_hour`, `photo_attached` | Weak in both periods (strength < 0.05 overall and post-May) |

`log_amount` and `small_claim` are also correlated (0.83) but both are kept: `small_claim` marks the policy threshold, which matters after May even though raw amount does not.

In [ ]:
DROP = ["p_past_frauds", "days_since_purchase", "serial_prior_claims", "submit_hour", "photo_attached"]
NUMERIC_FINAL = [c for c in CANDIDATES if c not in DROP]
FEATURES_FINAL = NUMERIC_FINAL + CATEGORICAL

X_train, y_train = train_fe[FEATURES_FINAL], train_fe["is_fraud"]
X_test = test_fe[FEATURES_FINAL]

assert X_train.columns.equals(X_test.columns)
assert X_train.isna().sum().sum() == 0 and X_test.isna().sum().sum() == 0
print(f"Final features: {len(FEATURES_FINAL)} ({len(NUMERIC_FINAL)} numeric, {len(CATEGORICAL)} categorical)")
print(f"X_train {X_train.shape} | X_test {X_test.shape} | fraud in train: {y_train.sum()} ({y_train.mean():.2%})")

In [ ]:
def feature_group(col):
    if col in CATEGORICAL:
        return "categorical"
    if col.startswith("p_past") or col == "p_frauds_90d":
        return "partner fraud history"
    if col.startswith("p_"):
        return "partner behaviour"
    if col in ("partner_tenure_days", "partner_is_new", "new_x_post"):
        return "partner profile"
    return "claim"

for group, cols in pd.Series(FEATURES_FINAL).groupby(pd.Series(FEATURES_FINAL).map(feature_group)):
    print(f"{group:<22} ({len(cols):>2}): {', '.join(cols)}")

### 6.8 Feature engineering summary

- **30 features** go into modelling: 25 numeric, 5 categorical. All are known at claim submission time; no future information is used.
- **Strongest signals:** partner fraud history, the post-May new-partner interaction, partner tenure and recent partner claim volume.
- **Signal drift after May 2026:** claim-amount features lose their power. The model must be **validated on post-May claims (May–Jun 2026)**, the period that looks like the test set, not on a random split.
- **Next step:** modelling. Train on claims before May 2026, validate on May–Jun 2026, and judge by **Rs of fraud caught in the top 40 claims a month**, not accuracy.

## 7. Modelling

### 7.1 Validation design

The validation has to look like the real test situation:

| | Model trained on labels up to | Scored claims | Review capacity |
|---|---|---|---|
| **Test (real)** | 30 Jun 2026 | Jul–Sep 2026 | 40 / month |
| **Fold A** | 30 Apr 2026 | May–Jun 2026 | 40 / month |
| **Fold B (main)** | 31 May 2026 | Jun 2026 | 40 / month |

Rules:
1. **Time split, never random.** Train on earlier claims, validate on later ones.
2. **History frozen at the cutoff.** Label-based partner history for validation claims uses only outcomes known before the cutoff, plus the 30-day label lag used everywhere (Section 6).
3. **Decisions are made on Fold B only, by rules fixed in advance** (below). After the configuration is frozen (7.7), June is not used again for tuning. The Jul–Sep test set is never used for model selection.

**Metric:** frauds caught and **Rs of fraud caught in the top 40 claims per month**, and Rs caught per claim reviewed. PR-AUC is a secondary ranking metric. Accuracy is reported only to show why it misleads.

In [ ]:
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.stats.proportion import proportion_confint
from xgboost import XGBClassifier

REVIEWS_PER_MONTH = 40
GOODWILL_RS = 380
CONTACT_RS = 260


def add_interactions(df):
    df["new_x_post"] = df["partner_is_new"] * df["after_may_change"]
    df["near_thr_x_uninsp"] = df["near_threshold"] * (1 - df["partner_inspected"])
    for col in CATEGORICAL:
        df[col] = df[col].astype("category")
    return df


def make_fold(cutoff):
    """Train on claims before `cutoff`, validate on claims after it.
    Labels from `cutoff` onwards are hidden while building features, so history is frozen at the cutoff.
    Open claims stay in for the behaviour features; the merge with `train_clean` keeps labelled claims only."""
    cutoff = pd.Timestamp(cutoff)
    masked = train_all_clean.copy()
    masked.loc[masked["submitted_at"] >= cutoff, "is_fraud"] = np.nan
    fe = add_interactions(build_features(masked, test_clean))
    fe = (fe[fe["dataset"] == "train"].drop(columns="is_fraud")
            .merge(train_clean[["claim_id", "is_fraud"]], on="claim_id"))
    return fe[fe["submitted_at"] < cutoff].copy(), fe[fe["submitted_at"] >= cutoff].copy()


def make_logistic():
    prep = ColumnTransformer([("num", StandardScaler(), NUMERIC_FINAL),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return make_pipeline(prep, LogisticRegression(C=0.1, class_weight="balanced", max_iter=3000))


def make_xgb():
    return XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=2, subsample=0.8, colsample_bytree=0.8,
                         min_child_weight=3, scale_pos_weight=20, enable_categorical=True,
                         tree_method="hist", random_state=42)


def fit_model(kind, tr, post_may_weight):
    weights = np.where(tr["after_may_change"] == 1, post_may_weight, 1.0)
    if kind == "logistic":
        return make_logistic().fit(tr[FEATURES_FINAL], tr["is_fraud"], logisticregression__sample_weight=weights)
    return make_xgb().fit(tr[FEATURES_FINAL], tr["is_fraud"], sample_weight=weights)


def score(model, df):
    """Fraud-risk score = predicted probability of class 1 (is_fraud = 1)."""
    assert list(model.classes_) == [0, 1]
    return model.predict_proba(df[FEATURES_FINAL])[:, 1]


def top_k(df, s, k=REVIEWS_PER_MONTH):
    """Review the k highest-scoring claims in df. Returns counts and rupees."""
    picked = df.assign(score=s).nlargest(k, "score")
    caught = int(picked["is_fraud"].sum())
    rs = picked.loc[picked["is_fraud"] == 1, "claim_amount_inr"].sum()
    return {"K": k, "fraud caught": f"{caught} / {int(df['is_fraud'].sum())}", "precision@K": caught / k,
            "Rs fraud caught": rs, "Rs / review": round(rs / k), "_caught": caught, "_ids": set(picked["claim_id"])}


def wilson(k, n):
    lo, hi = proportion_confint(k, n, method="wilson")
    return f"{k}/{n} = {k / n:.0%} (95% CI {lo:.0%}–{hi:.0%})"

In [ ]:
tr_a, va_a = make_fold("2026-05-01")
tr_b, va_b = make_fold("2026-06-01")
pd.DataFrame({
    "train claims": [len(tr_a), len(tr_b)], "train frauds": [tr_a["is_fraud"].sum(), tr_b["is_fraud"].sum()],
    "of which post-May": [tr_a.query("after_may_change == 1")["is_fraud"].sum(), tr_b.query("after_may_change == 1")["is_fraud"].sum()],
    "valid claims": [len(va_a), len(va_b)], "valid frauds": [va_a["is_fraud"].sum(), va_b["is_fraud"].sum()],
}, index=["Fold A: valid May-Jun", "Fold B: valid Jun"])

### 7.2 Fold A diagnosis: regime inversion or technical error?

Models trained only on pre-May claims score ROC-AUC far **below** 0.5 on May–June. Before reading anything into this, rule out a technical error.

In [ ]:
model_a = fit_model("logistic", tr_a, post_may_weight=1)   # no post-May rows in Fold A training, weight has no effect
s_a = score(model_a, va_a)
y_a = va_a["is_fraud"].values
n_pos, n_neg = y_a.sum(), len(y_a) - y_a.sum()

# Point-in-time check: recompute p_past_frauds by brute force for 200 validation claims
known = train_all_clean[train_all_clean["is_fraud"].notna() & (train_all_clean["submitted_at"] < "2026-05-01")]
sample = va_a.sample(200, random_state=0)
brute = [known.loc[(known["partner_id"] == p) & (known["submitted_at"] < t - pd.Timedelta(days=LABEL_LAG_DAYS)), "is_fraud"].sum()
         for p, t in zip(sample["partner_id"], sample["submitted_at"])]

checks = {
    "Train dates all before 1 May 2026": tr_a["submitted_at"].max() < pd.Timestamp("2026-05-01"),
    "Validation dates all from 1 May 2026": va_a["submitted_at"].min() >= pd.Timestamp("2026-05-01"),
    "is_fraud encoded as 0/1 (1 = fraud)": set(tr_a["is_fraud"].unique()) == {0, 1} and set(y_a) == {0, 1},
    "predict_proba column 1 = P(is_fraud = 1)": list(model_a.classes_) == [0, 1],
    "sklearn ROC-AUC = Mann-Whitney U / (n_pos * n_neg)":
        np.isclose(roc_auc_score(y_a, s_a), mannwhitneyu(s_a[y_a == 1], s_a[y_a == 0]).statistic / (n_pos * n_neg)),
    "Same category levels in train and validation": all(list(tr_a[c].cat.categories) == list(va_a[c].cat.categories) for c in CATEGORICAL),
    "History uses only labels before cutoff and 30-day lag (brute force, 200 rows)": np.allclose(brute, sample["p_past_frauds"]),
}
display(pd.Series(checks, name="passed").to_frame())
print(f"ROC-AUC (score)  = {roc_auc_score(y_a, s_a):.3f}")
print(f"ROC-AUC (-score) = {roc_auc_score(y_a, -s_a):.3f}   <- the ranking is inverted, not random")
print(f"In-sample ROC-AUC on pre-May training data = {roc_auc_score(tr_a['is_fraud'], score(model_a, tr_a)):.3f}")

In [ ]:
print("Highest-scoring May-Jun claims under the pre-May model:")
display(va_a.assign(score=s_a).nlargest(8, "score")[["claim_amount_inr", "partner_tenure_days", "partner_is_new",
                                                      "p_past_fraud_rate", "score", "is_fraud"]].round(3))

flip = ["partner_tenure_days", "partner_is_new", "small_claim", "amount_to_price", "log_amount",
        "customer_prior_claims", "p_past_fraud_rate", "p_claims_90d"]
pd.DataFrame({
    "AUC pre-May (train)": [roc_auc_score(tr_a["is_fraud"], tr_a[f]) for f in flip],
    "AUC May-Jun (valid)": [roc_auc_score(va_a["is_fraud"], va_a[f]) for f in flip],
}, index=flip).round(3)

**Result: genuine regime inversion, not a bug.** All technical checks pass, and reversing the score gives ROC-AUC ≈ 0.87: the model ranks claims in a consistent order, just the wrong way round for the new period.

The single-feature AUCs show why. Before May, fraud went with **long-tenured partners and large claims** (`partner_tenure_days` AUC 0.64, `amount_to_price` 0.79). From May it goes with **first-year partners and small claims** (`partner_tenure_days` 0.13, `partner_is_new` 0.91). The pre-May model learned the old pattern and puts the old pattern's claims (large claims from partners with 3+ years' tenure) at the top, and none of them are fraud. Partner history cannot rescue it: the outlets behind the post-May fraud had no confirmed fraud before May.

This result is kept as evidence of a severe regime shift. It is why the main validation (Fold B) lets the model see May's labels before scoring June, as the real model will see May–June before scoring Jul–Sep.

### 7.3 Post-May weighting

**Where 5× came from:** it was picked during early prototyping after looking at June results, not from a controlled test. That is a form of tuning on June, so it is re-checked here.

**Controlled comparison** on Fold B, weights 1×, 2×, 3×, 5×, everything else fixed.
**Decision rule, set before looking at the table:** pick the weight with the most frauds caught in the top 40; break ties by Rs caught, then by the **smaller** weight (less tuning).

In [ ]:
weight_rows = []
for kind in ["logistic", "xgb"]:
    for w in [1, 2, 3, 5]:
        s = score(fit_model(kind, tr_b, w), va_b)
        t = top_k(va_b, s)
        weight_rows.append({"model": kind, "post-May weight": w, "fraud caught@40": t["_caught"],
                            "Rs caught@40": t["Rs fraud caught"], "PR-AUC": average_precision_score(va_b["is_fraud"], s),
                            "ROC-AUC": roc_auc_score(va_b["is_fraud"], s)})
weights_table = pd.DataFrame(weight_rows)

def pick_weight(rows):
    return rows.sort_values(["fraud caught@40", "Rs caught@40", "post-May weight"],
                            ascending=[False, False, True]).iloc[0]["post-May weight"]

CHOSEN_WEIGHT = {kind: int(pick_weight(weights_table[weights_table["model"] == kind])) for kind in ["logistic", "xgb"]}
print("Chosen post-May weight:", CHOSEN_WEIGHT)
weights_table.round(3)

**Result**
- **Logistic regression:** up-weighting post-May claims helps (11 frauds caught at 1× vs 14 at 3× and 5×; PR-AUC rises steadily). 3× and 5× tie on frauds and rupees, so the rule picks **3×**.
- **XGBoost:** weighting makes no consistent difference (13–14 frauds at every weight). The rule picks 5× on rupees, by a margin of one claim's value.
- The differences between 2×, 3× and 5× are within noise for 22 frauds. The direction (some up-weighting helps the linear model) is the robust part, not the exact value.

### 7.4 Logistic regression vs XGBoost (Fold B, June)

Both catch 14 of 22 in the top 40. Does XGBoost add ranking value?

In [ ]:
s_log = score(fit_model("logistic", tr_b, CHOSEN_WEIGHT["logistic"]), va_b)
s_xgb = score(fit_model("xgb", tr_b, CHOSEN_WEIGHT["xgb"]), va_b)
s_blend = (pd.Series(s_log).rank().values + pd.Series(s_xgb).rank().values) / 2   # rank average, for reference

rows = []
for name, s in [("Logistic", s_log), ("XGBoost", s_xgb), ("Rank blend (reference)", s_blend)]:
    row = {"model": name, "PR-AUC": round(average_precision_score(va_b["is_fraud"], s), 3)}
    for k in (40, 80, 120):
        t = top_k(va_b, s, k)
        row[f"caught@{k}"] = t["fraud caught"]
        row[f"Rs@{k}"] = t["Rs fraud caught"]
        row[f"Rs/review@{k}"] = t["Rs / review"]
    row["precision@40"] = f"{top_k(va_b, s, 40)['precision@K']:.0%}"
    rows.append(row)
pd.DataFrame(rows).set_index("model")

In [ ]:
ids_log, ids_xgb = top_k(va_b, s_log)["_ids"], top_k(va_b, s_xgb)["_ids"]
both = ids_log & ids_xgb
fraud_ids = set(va_b.loc[va_b["is_fraud"] == 1, "claim_id"])
union = va_b["claim_id"].isin(ids_log | ids_xgb).values
print(f"Top-40 overlap: {len(both)} of 40 claims")
print(f"Frauds caught by both: {len(both & fraud_ids)} | only logistic: {len((ids_log - ids_xgb) & fraud_ids)} | "
      f"only XGBoost: {len((ids_xgb - ids_log) & fraud_ids)}")
print(f"Spearman correlation of scores: all June claims {spearmanr(s_log, s_xgb)[0]:.2f} | "
      f"within the top-40 union {spearmanr(s_log[union], s_xgb[union])[0]:.2f}")

**Result: XGBoost adds no reliable ranking value.**
- At the desk's capacity (top 40) both catch 14 frauds and the same rupees. Each catches 3 frauds the other misses.
- XGBoost has the higher PR-AUC: it places a few frauds slightly higher at the very top. But going deeper, logistic regression catches more (18 vs 16 at top 80, 20 vs 17 at top 120).
- The two models agree on the overall order (Spearman 0.71) but disagree on the order *within* the top claims, so neither is consistently better.
- The rank blend catches one more fraud at top 40, which is within noise and would be a choice made by looking at June. It is not adopted.
- **Choice: logistic regression.** It is equal at capacity, better deeper in the list, and simpler to explain.

### 7.5 Monthly validation table (K = 40)

Fold A scores May and June with a model that has seen no post-May data; Fold B scores June after seeing May.

In [ ]:
def monthly_rows(fold, va, scores):
    out = []
    month = va["submitted_at"].dt.to_period("M").astype(str)
    for name, s in scores.items():
        for m in sorted(month.unique()):
            mask = (month == m).values
            t = top_k(va[mask], np.asarray(s)[mask])
            out.append({"Fold": fold, "Model": name, "Month": m, "Top-K": t["K"], "Fraud caught": t["fraud caught"],
                        "Precision@K": f"{t['precision@K']:.0%}", "Rs fraud caught": t["Rs fraud caught"],
                        "Rs / review": t["Rs / review"]})
    return out

rule_recent = lambda v: (v["p_frauds_90d"] + v["p_past_fraud_rate"]).values
rule_new = lambda v: (v["partner_is_new"] - v["claim_amount_inr"] / 1e6).values

monthly = pd.DataFrame(
    monthly_rows("A", va_a, {
        "Logistic": score(fit_model("logistic", tr_a, CHOSEN_WEIGHT["logistic"]), va_a),
        "XGBoost": score(fit_model("xgb", tr_a, CHOSEN_WEIGHT["xgb"]), va_a),
        "Rule: recent-fraud partners": rule_recent(va_a), "Rule: new partners first": rule_new(va_a)})
    + monthly_rows("B", va_b, {
        "Logistic": s_log, "XGBoost": s_xgb,
        "Rule: recent-fraud partners": rule_recent(va_b), "Rule: new partners first": rule_new(va_b)}))
monthly.set_index(["Fold", "Model", "Month"])

**Result**
- **Fold A:** both models catch **0 frauds** in May and in June. Only the "new partners first" rule finds any (4 in May, 6 in June), because it happens to match the new pattern.
- **Fold B (June):** logistic regression and XGBoost catch 14 of 22 (Rs 19,765 and Rs 19,839; about Rs 495 per review), ahead of the recent-fraud rule (10) and the new-partner rule (6).
- One month of post-May labels (May) is enough to turn the model around. The real model will have two (May–June).

### 7.6 Uncertainty

June has only **22 frauds**. The June figures are observed counts for one month, not precise estimates of future performance. Wilson 95% confidence intervals:

In [ ]:
t = top_k(va_b, s_log)
june_fraud_rs = va_b.loc[va_b["is_fraud"] == 1, "claim_amount_inr"].sum()
print("Logistic regression, June, top 40:")
print("  Precision@40 (share of reviewed claims that are fraud):", wilson(t["_caught"], 40))
print("  Recall (share of June frauds caught):                  ", wilson(t["_caught"], int(va_b["is_fraud"].sum())))
print(f"  Rupees caught: Rs {t['Rs fraud caught']:,.0f} of Rs {june_fraud_rs:,.0f} June fraud "
      f"({t['Rs fraud caught'] / june_fraud_rs:.0%}); one missed claim alone is Rs 17,866")

**Reading:** at 40 reviews, between roughly 1 in 5 and 1 in 2 reviewed claims would be fraud, and the model would catch somewhere between ~40% and ~80% of fraud cases. The rupee figure is even less stable: a single large missed claim (Rs 17,866) is 39% of June's fraud value.

### 7.7 Precision-recall curve (Fold B)

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(figsize=(8, 5))
for name, s in [("Logistic", s_log), ("XGBoost", s_xgb),
                ("Rule: recent-fraud partners", rule_recent(va_b)), ("Rule: new partners first", rule_new(va_b))]:
    p, r, _ = precision_recall_curve(va_b["is_fraud"], s)
    ax.step(r, p, where="post", label=f"{name} (AP = {average_precision_score(va_b['is_fraud'], s):.3f})")
ax.axhline(va_b["is_fraud"].mean(), color="grey", ls=":", label=f"Random (AP = {va_b['is_fraud'].mean():.3f})")
t = top_k(va_b, s_log)
ax.scatter(t["_caught"] / va_b["is_fraud"].sum(), t["precision@K"], s=80, color="black", zorder=5, label="Logistic, top 40")
ax.set(xlabel="Recall", ylabel="Precision", title="Precision-recall, June 2026 (Fold B)", xlim=(0, 1.02), ylim=(0, 1.05))
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### 7.8 Frozen configuration

The configuration below is now **fixed**. June is not used for any further tuning, and the Jul–Sep test set is never used for selection.

| Item | Choice | Why |
|---|---|---|
| Model | Logistic regression (C = 0.1, class-balanced) | Equal to XGBoost at top 40, better deeper, simpler |
| Features | 30 (`FEATURES_FINAL`, Section 6.7) | Unchanged since feature engineering |
| Label lag | 30 days (`LABEL_LAG_DAYS`) | Same history staleness in train and test |
| Post-May weight | 3× | Controlled comparison (7.3), pre-set decision rule |
| Training data | All labelled claims to 30 Jun 2026 (11,146) | Includes both post-May months |
| Output score | Raw model fraud-risk score, **no calibration** | See 8.1 |

In [ ]:
FINAL_CONFIG = {"model": "logistic", "post_may_weight": CHOSEN_WEIGHT["logistic"], "label_lag_days": LABEL_LAG_DAYS,
                "features": FEATURES_FINAL, "training_cutoff": "2026-06-30"}
assert FINAL_CONFIG["post_may_weight"] == 3
print({k: (len(v) if k == "features" else v) for k, v in FINAL_CONFIG.items()})

## 8. Final model and outputs

### 8.1 Retrain on all labelled claims

**Score scale.** Class weighting (balanced classes, post-May 3×) inflates the model's raw outputs: in the test set 403 claims score above 0.5. A scorer that thresholds at 0.5 (accuracy, F1) would then see hundreds of false alarms, although the *ranking* is fine.

The raw score is therefore passed through a **prior correction** that undoes the training weights with a formula: divide the predicted odds by the factor the weights inflated the training odds by. This is:
- **monotone**: the ranking, ROC-AUC and PR-AUC are exactly unchanged;
- **label-free**: nothing is fitted on June or any other outcomes (unlike the Platt scaling dropped earlier, which was not independent of training).

On Fold B the same correction gives a mean score of 3.3% against June's actual fraud rate of 3.1%, and accuracy at a 0.5 threshold of 96.4% instead of 88.1%. The score is a fraud-risk score: higher means more likely fraudulent.

In [ ]:
def weight_inflation(train_df, post_may_weight):
    """How much class + sample weights inflated the training odds of fraud vs the true odds."""
    y = train_df["is_fraud"].values
    sample_w = np.where(train_df["after_may_change"] == 1, post_may_weight, 1.0)
    n, n_pos = len(y), y.sum()
    class_w = np.where(y == 1, n / (2 * n_pos), n / (2 * (n - n_pos)))     # sklearn "balanced"
    w = class_w * sample_w
    return (w[y == 1].sum() / w[y == 0].sum()) / (n_pos / (n - n_pos))


def prior_correct(p, inflation):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    odds = p / (1 - p) / inflation
    return odds / (1 + odds)


final_model = fit_model(FINAL_CONFIG["model"], train_fe, FINAL_CONFIG["post_may_weight"])
raw_test_score = score(final_model, test_fe)
inflation = weight_inflation(train_fe, FINAL_CONFIG["post_may_weight"])
test_score = prior_correct(raw_test_score, inflation)

assert spearmanr(raw_test_score, test_score)[0] > 0.9999   # ranking unchanged
print(f"Trained on {len(train_fe):,} labelled claims ({train_fe['is_fraud'].sum()} fraud, "
      f"{train_fe.query('after_may_change == 1')['is_fraud'].sum()} post-May), post-May weight {FINAL_CONFIG['post_may_weight']}x")
print(f"Odds inflation from training weights: {inflation:.1f}x")
print(f"Raw score       : median {np.median(raw_test_score):.3f} | above 0.5: {(raw_test_score > 0.5).sum()}")
print(f"Corrected score : median {np.median(test_score):.4f} | mean {test_score.mean():.2%} | above 0.5: {(test_score > 0.5).sum()}")

### 8.2 Monthly review list (top 40 per month)

In [ ]:
# The package (`python -m kestrel_fraud.train`) writes the deliverables to outputs/. The notebook writes its own
# copies to outputs/notebook/ so re-running it never overwrites them; section 8.3 checks the two agree.
OUTPUT_DIR = DATA_DIR.parent / "outputs" / "notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_COLUMNS = [
    "claim_id", "submitted_at", "month", "score", "rank_in_month", "claim_amount_inr",
    "partner_id", "partner_is_new", "partner_tenure_days", "partner_type", "city",
    # risk features
    "p_frauds_90d", "p_past_fraud_rate", "p_claims_30d", "p_claims_90d", "new_x_post",
    "near_threshold", "near_thr_x_uninsp", "partner_inspected", "amount_to_price", "customer_prior_claims",
]
scored = test_fe.assign(score=test_score, month=test_fe["submitted_at"].dt.strftime("%Y-%m"))
review_list = (scored.sort_values("score", ascending=False).groupby("month").head(REVIEWS_PER_MONTH)
                     .sort_values(["month", "score"], ascending=[True, False]))
review_list["rank_in_month"] = review_list.groupby("month").cumcount() + 1
review_list = review_list[REVIEW_COLUMNS].reset_index(drop=True)

assert "is_fraud" not in review_list.columns
assert review_list.groupby("month").size().eq(REVIEWS_PER_MONTH).all() and review_list["month"].nunique() == 3
review_list.to_csv(OUTPUT_DIR / "review_list_jul_sep_2026.csv", index=False)
print("Saved:", OUTPUT_DIR / "review_list_jul_sep_2026.csv", review_list.shape)

review_list.groupby("month").agg(
    claims=("claim_id", "size"), partners=("partner_id", "nunique"),
    share_new_partner=("partner_is_new", "mean"), with_recent_partner_fraud=("p_frauds_90d", lambda s: (s > 0).mean()),
    median_amount_rs=("claim_amount_inr", "median"), lowest_score=("score", "min"),
).round(2)

In [ ]:
partner_view = (review_list.groupby("partner_id")
                           .agg(claims_in_list=("claim_id", "size"), partner_is_new=("partner_is_new", "max"),
                                tenure_days=("partner_tenure_days", "min"), max_recent_frauds=("p_frauds_90d", "max"))
                           .sort_values("claims_in_list", ascending=False))
new_with_test_claims = test_fe.loc[test_fe["partner_is_new"] == 1, "partner_id"].nunique()
print(f"{len(partner_view)} partners in the list; top 7 hold {partner_view['claims_in_list'].head(7).sum() / len(review_list):.0%} of the 120 reviews")
print(f"First-year partners with test claims: {new_with_test_claims}; in the list: {(partner_view['partner_is_new'] == 1).sum()}")
partner_view.head(10)

### 8.3 Predictions file

`predictions.csv`: one row per test claim, in `sample_submission.csv` shape. Higher score = more likely fraudulent. The deliverable is written by the package; this cell checks the notebook's copy and that both give the same scores.

In [ ]:
predictions = pd.DataFrame({"claim_id": test_fe["claim_id"], "score": test_score})
predictions = sample_submission[["claim_id"]].merge(predictions, on="claim_id", how="left")

results = {}
for name in ["predictions.csv"]:
    predictions.to_csv(OUTPUT_DIR / name, index=False, float_format="%.10g")
    saved = pd.read_csv(OUTPUT_DIR / name)
    results[name] = {
        "same row count as test": len(saved) == len(test) == len(sample_submission),
        "same claim IDs as test": set(saved["claim_id"]) == set(test["claim_id"]),
        "same order as sample_submission": saved["claim_id"].equals(sample_submission["claim_id"]),
        "no duplicate claim IDs": saved["claim_id"].is_unique,
        "no missing scores": saved["score"].notna().all(),
        "exact column names": list(saved.columns) == list(sample_submission.columns),
        "no index column": saved.shape[1] == 2,
        "scores within [0, 1]": saved["score"].between(0, 1).all(),
        "ranking preserved after save": spearmanr(saved["score"], predictions["score"])[0] > 0.9999,
    }
checks = pd.DataFrame(results)
assert checks.values.all(), checks
print("Saved:", OUTPUT_DIR / "predictions.csv")

package_file = DATA_DIR.parent / "outputs" / "predictions.csv"
if package_file.exists():
    package = pd.read_csv(package_file)
    gap = (package["score"] - predictions["score"]).abs().max()
    print(f"Package predictions.csv: same claim order {package['claim_id'].equals(predictions['claim_id'])}, "
          f"largest score difference {gap:.1e}")
else:
    print("Package predictions.csv not found: run python -m kestrel_fraud.train")
checks

### 8.4 What we expect the predictions to score

Written **before** the outcomes are known. The only evidence is Fold B (June 2026, 22 frauds), plus how the test period differs from it.

In [ ]:
month = test_fe["submitted_at"].dt.strftime("%Y-%m")
expectation_inputs = pd.DataFrame({
    "claims": test_fe.groupby(month).size(),
    "expected fraud rate (mean score)": pd.Series(test_score).groupby(month.values).mean(),
    "share new partner x post-May": test_fe.groupby(month)["new_x_post"].mean(),
    "share with visible recent partner fraud": test_fe.groupby(month)["p_frauds_90d"].apply(lambda s: (s > 0).mean()),
    "claims from partners new to the data": test_fe.groupby(month)["partner_id"].apply(lambda s: (~s.isin(train_fe["partner_id"])).sum()),
})
june_ref = pd.Series({
    "claims": len(va_b), "expected fraud rate (mean score)": va_b["is_fraud"].mean(),
    "share new partner x post-May": va_b["new_x_post"].mean(),
    "share with visible recent partner fraud": (va_b["p_frauds_90d"] > 0).mean(), "claims from partners new to the data": np.nan,
}, name="June (Fold B, actual rate)")
pd.concat([expectation_inputs, june_ref.to_frame().T]).round(3)

**Expected scores**

| Metric | June (Fold B, observed) | Expected on Jul–Sep test | Why |
|---|---|---|---|
| PR-AUC / average precision (primary) | 0.37 (random 0.03) | **best guess 0.27, range 0.20–0.35** | Staleness backtest pools to 0.285; small extra cut for unmeasured drift |
| ROC-AUC | 0.91 | **best guess 0.87, range 0.80–0.90** | Backtest pools to 0.90; same cut |
| Precision in top 40 per month (120 reviewed) | 35% (14 / 40) | **25–35%** (≈32 frauds in 120) | Backtest: 14 in a current month, 10 in a stale one |
| Accuracy at a 0.5 threshold | 96.4% | **~95–96.5%** | Close to "flag nothing"; unlikely to exceed 97% |

**Why lower than June**
1. **History goes stale (measured).** Test claims only see outcomes to 30 June, so August and September claims are scored without any July–August investigation results. `python -m kestrel_fraud.evaluate` re-scores the June claims with history frozen 1 and 2 months earlier: PR-AUC falls from 0.371 to 0.252 and 0.253 (top-40 catch 14 → 10). Pooled over July/August/September-like scoring: PR-AUC 0.285, ROC-AUC 0.902 (`evidence/validation_report.md`).
2. **New outlets with no record.** First-year partners make up a growing share of claims (about 15% in July to 20% in September), and some test partners have no claims at all in the training data. If new fraud comes from outlets with no confirmed fraud yet, the model can only use behaviour and tenure signals, which are weaker.
3. **Only two months of the new pattern.** The model learned the post-May pattern from 36 frauds. Fold A showed that when the pattern changes, the ranking can invert completely.

**Why not much lower**
- The seven outlets that drive the review list (named in the review list output) all have confirmed May–June fraud, and they keep submitting claims in the test period.
- Tenure, claim volume and small/uninspected-claim signals worked on June without needing history (the new-partner rule alone reached ROC-AUC 0.89).

**Downside scenario:** if the fraudsters change again (new outlets, or a return to large claims), ROC-AUC could fall to **0.6–0.7**, as Fold A showed on a smaller scale. **Accuracy is not a useful score here:** with a fraud rate of about 3%, flagging nothing already scores about 97%.

## 9. Modelling report

### Validation strategy
- Time-based validation only. The test period (Jul–Sep 2026) lies entirely after the 1 May 2026 rule that auto-approves claims under Rs 2,000, so validation must use post-May claims.
- Partner fraud history uses only outcomes at least 30 days old, in training and scoring alike, and is frozen at each validation cutoff.
- Decisions were taken on Fold B (train to 31 May, validate June) with rules set in advance. The configuration was then frozen. The test set was not used for selection.

### Fold A: regime shift
A model trained only on pre-May claims ranks May–June claims **backwards** (ROC-AUC 0.13; reversed, 0.87). Technical checks (dates, labels, probability column, AUC computation, categories, point-in-time history) all pass. Before May, fraud was associated with large claims from long-standing partners; from May it is associated with small claims from first-year partners. This is the clearest evidence in the data that the fraud pattern changed with the policy.

### Fold B: June result
With May's labels available, the logistic model caught **14 of 22** June frauds in its top 40 (precision 35%, 95% CI 22–50%; recall 64%, CI 43–80%), worth **Rs 19,765, about Rs 494 per review**. These are observed counts for one month with 22 frauds, not precise forecasts.

### Model comparison
| June, top 40 | Frauds | Rs caught | Rs / review |
|---|---|---|---|
| Logistic regression (3×) | 14 / 22 | 19,765 | 494 |
| XGBoost (5×) | 14 / 22 | 19,839 | 496 |
| Rule: partners with recent fraud | 10 / 22 | 14,538 | 363 |
| Rule: new partners first | 6 / 22 | 5,481 | 137 |
| Baseline: flag nothing | 0 / 22 | 0 | – |

XGBoost adds no reliable value: equal at the top 40, behind at the top 80 and 120.

### Post-May weighting
The original 5× was an informal choice made while looking at June. A controlled comparison (1×, 2×, 3×, 5×) showed that up-weighting helps the logistic model (11 → 14 frauds at top 40). A rule fixed in advance chose **3×**. Values from 2× to 5× are within noise of each other.

### Why accuracy is not sufficient
Fraud is about 3% of June claims, so flagging **nothing** scores 96.9% accuracy. The model reviewing 40 claims scores 95.2% while catching 14 frauds. Accuracy rises when fewer claims are flagged, whatever their value. A ">97% accuracy" target can only be met by under-using the review desk. The business measure is rupees of fraud caught per claim reviewed.

### Top-40 economics (June, observed)
- 40 reviews: Rs 19,765 fraud caught; 26 genuine customers held (Rs 380 goodwill each = Rs 9,880).
- Net after goodwill: **+Rs 9,885**. Net after also charging Rs 260 per review contact: **−Rs 515** (roughly break-even). Policy §4 does not say whether the contact cost applies to every review.

### Ritu's hypothesis (see `Ritu_Hypothesis_New_Partners.ipynb`)
Being a newer partner is **associated** with fraud only after May 2026, and the association comes mainly from five outlets. Newer partners as a group are not more likely to have any fraud than established ones. The data do not show that being new causes fraud; the new partners and the policy change arrived at the same time.

### Final model
Logistic regression, 30 features, 30-day label lag, post-May weight 3×, trained on all 11,146 labelled claims to 30 Jun 2026. Predictions (`outputs/predictions.csv`) are the model's fraud-risk score after a label-free prior correction that undoes the training weights (ranking unchanged). Expected test PR-AUC about 0.27 (range 0.20–0.35), ROC-AUC about 0.87 and top-40 precision 25–35% (Section 8.4). The review list takes the top 40 per month.

### Limitations
- Only two months (and 36 frauds) of the post-May pattern exist. The test period could shift again.
- June results rest on 22 frauds; confidence intervals are wide.
- Some legacy Zoho "0" labels may be open cases.
- Scores are on a probability-like scale (mean close to the observed fraud rate) but are not fitted to outcomes; treat them as risk rankings.
- The review list contains only first-year partners' claims. Without some random reviews of established partners, a new shift would go unseen.
- Four claim descriptions contained text aimed at automated analysis tools (removed in cleaning); this should be raised with IT.